# Lab 02 - Clusters y Optimización

**Objetivos:**
- Probar autoscaling con diferentes cargas
- Calcular costos de diferentes configuraciones
- Optimizar configuración de clusters

## Ejercicio 1: Test de Autoscaling - Carga Ligera

In [ ]:
# =============================================================================
# TEST DE AUTOSCALING - CARGA LIGERA
# =============================================================================
# Objetivo: Verificar que el cluster NO escala para cargas pequeñas
# Con 1 millón de registros, el cluster debería mantener workers mínimos

# Importaciones
from pyspark.sql.functions import rand  # Genera números aleatorios
import time                              # Para medir tiempo de ejecución

print("🔹 Carga Ligera: Procesando 1 millón de registros")
start = time.time()  # Timestamp de inicio

# Crear DataFrame sintético de 1 millón de filas
# spark.range(start, end): Genera DataFrame con columna 'id' (números secuenciales)
df_light = spark.range(0, 1_000_000) \
    .withColumn(
        "value",                    # Nueva columna con valores aleatorios
        rand() * 1000               # rand(): número random entre 0-1, multiplicado por 1000
    ) \
    .withColumn(
        "category",                 # Nueva columna con categorías (0-9)
        (rand() * 10).cast("int")  # cast("int"): convierte a entero (trunca decimales)
    )

# 💡 NOTA: spark.range() es muy eficiente porque no carga datos en memoria
# Se usa comúnmente para testing y generación de datos sintéticos

# Realizar agregación simple: contar registros por categoría
# collect(): ACCIÓN que trae TODOS los resultados al driver
# ⚠️ Solo usar collect() con resultados pequeños (aquí son ~10 filas)
result = df_light.groupBy("category").count().collect()

end = time.time()  # Timestamp de fin
print(f"✅ Completado en {end - start:.2f} segundos")
print(f"📊 Resultados: {len(result)} categorías")

# 📈 QUÉ OBSERVAR EN SPARK UI:
# - Cluster → Metrics: Número de workers activos (debería mantenerse en mínimo)
# - Jobs: Tiempo de ejecución y número de tasks
# - No debería activarse autoscaling porque la carga es ligera

## Ejercicio 2: Test de Autoscaling - Carga Pesada

In [ ]:
# =============================================================================
# TEST DE AUTOSCALING - CARGA PESADA
# =============================================================================
# Objetivo: Forzar autoscaling incrementando carga de trabajo
# Con 50 millones de registros y repartición, el cluster DEBE escalar

print("🔸 Carga Pesada: Procesando 50 millones de registros")
start = time.time()

# Crear DataFrame grande de 50 millones de filas
df_heavy = spark.range(0, 50_000_000) \
    .withColumn("value", rand() * 1000) \
    .withColumn("category", (rand() * 100).cast("int")) \
    .repartition(16)  # ← CLAVE: Fuerza redistribución en 16 particiones

# 🔄 ¿QUÉ HACE repartition(16)?
# - Redistribuye los datos en 16 particiones iguales
# - Aumenta el paralelismo (más tareas concurrentes)
# - Provoca SHUFFLE: movimiento de datos entre executors (costoso pero necesario)
# - Útil cuando hay muchos workers pero pocas particiones (bottleneck)

# ⚖️ TRADE-OFF de repartition():
# ✅ Pro: Más paralelismo = más cores trabajando simultáneamente
# ❌ Con: Shuffle tiene costo de I/O de red

# 📊 Uso típico: Repartir a (num_workers × cores_per_worker × 2-3)
# Ejemplo: 5 workers × 4 cores × 2 = 40 particiones

# Importar funciones de agregación estadística
from pyspark.sql.functions import avg, sum, stddev, min, max

# Realizar agregaciones complejas por categoría
result = df_heavy.groupBy("category").agg(
    avg("value").alias("avg_value"),      # Promedio
    sum("value").alias("sum_value"),      # Suma total
    stddev("value").alias("stddev_value"), # Desviación estándar (dispersión)
    min("value").alias("min_value"),      # Valor mínimo
    max("value").alias("max_value")       # Valor máximo
).collect()

end = time.time()

# 📊 AGREGACIONES EXPLICADAS:
# - avg: Suma todos los valores y divide entre el conteo
# - sum: Suma acumulativa de todos los valores
# - stddev: Mide qué tan dispersos están los datos del promedio
# - min/max: Valores extremos del conjunto

print(f"✅ Completado en {end - start:.2f} segundos")
print(f"📊 Resultados: {len(result)} categorías con estadísticas")
print(f"\n💡 Revisa la UI de Spark para ver el autoscaling en acción")

# 📈 QUÉ OBSERVAR EN SPARK UI:
# - Cluster → Metrics: Workers deberían INCREMENTAR de min a max
# - Jobs → Stages: Verás stage de shuffle (intercambio de datos)
# - Executors: Más executors activos procesando datos
# - Timeline: Gráfico mostrando cuándo se agregaron workers

## Ejercicio 3: Calculadora de Costos

In [ ]:
# =============================================================================
# CALCULADORA DE COSTOS DE DATABRICKS
# =============================================================================
# Objetivo: Estimar costos mensuales de diferentes configuraciones de cluster
# Incluye: Costo de VMs Azure + Costo de DBUs (Databricks Units)

def calcular_costo_cluster(vm_type, num_workers, horas_mes, tipo_cluster="all_purpose", usar_spot=False):
    """
    Calcula el costo mensual de un cluster de Databricks.
    
    El costo total tiene DOS componentes:
    1. VMs de Azure (infraestructura): CPU, RAM, Storage
    2. DBUs de Databricks (software): Gestión, optimizaciones, soporte
    
    Args:
        vm_type (str): Tipo de VM Azure (ej: "Standard_DS3_v2")
        num_workers (int): Número de workers en el cluster
        horas_mes (int): Horas de ejecución por mes
        tipo_cluster (str): "all_purpose" (interactivo) o "jobs" (automatizado)
        usar_spot (bool): True para usar Spot VMs (40% descuento)
    
    Returns:
        dict: Desglose completo de costos
    """
    # =========================================================================
    # CONFIGURACIÓN DE PRECIOS (East US, Mayo 2026)
    # =========================================================================
    # Precios de VMs Azure por hora (pueden variar por región)
    vm_precios = {
        "Standard_DS3_v2": {"cores": 4, "ram_gb": 14, "precio_hora": 0.192},
        "Standard_DS4_v2": {"cores": 8, "ram_gb": 28, "precio_hora": 0.384},
        "Standard_DS5_v2": {"cores": 16, "ram_gb": 56, "precio_hora": 0.768},
        "Standard_E4s_v3": {"cores": 4, "ram_gb": 32, "precio_hora": 0.252}
    }
    
    # Precios de DBUs (Databricks Units) - Premium tier
    # DBU = Unidad de procesamiento de Databricks
    dbu_precios = {
        "all_purpose": 0.55,  # Clusters interactivos (desarrollo, notebooks)
        "jobs": 0.22          # Clusters automatizados (50% más barato!)
    }
    
    # Obtener configuración de la VM seleccionada
    vm_config = vm_precios.get(vm_type, vm_precios["Standard_DS3_v2"])
    
    # =========================================================================
    # CÁLCULO DE DBUs
    # =========================================================================
    # Fórmula de DBUs: cores × 0.75 × num_workers
    # Factor 0.75: Ajuste de Databricks basado en capacidad efectiva de procesamiento
    dbus_por_hora = vm_config["cores"] * 0.75 * num_workers
    
    # =========================================================================
    # CÁLCULO DE COSTOS POR HORA
    # =========================================================================
    # Costo 1: VMs de Azure
    costo_vm_hora = vm_config["precio_hora"] * num_workers
    
    # Costo 2: DBUs de Databricks
    costo_dbu_hora = dbus_por_hora * dbu_precios[tipo_cluster]
    
    # Aplicar descuento de Spot VMs si está habilitado
    if usar_spot:
        # Spot VMs: Descuento promedio de 40% (puede variar por región/disponibilidad)
        # ⚠️ RIESGO: Pueden ser interrumpidas sin previo aviso por Azure
        # ✅ IDEAL: Workloads tolerantes a fallos (jobs batch, no interactivos)
        costo_vm_hora *= 0.60  # Pagar solo 60% del precio (40% descuento)
    
    # Costo total por hora = VM + DBU
    costo_total_hora = costo_vm_hora + costo_dbu_hora
    
    # Costo mensual = costo_hora × horas_mes
    costo_mensual = costo_total_hora * horas_mes
    
    print("💰 Calculadora de Costos de Cluster\n")
    
    # =========================================================================
    # RETORNAR DESGLOSE COMPLETO
    # =========================================================================
    return {
        "vm_type": vm_type,
        "workers": num_workers,
        "cores_total": vm_config["cores"] * num_workers,
        "ram_total_gb": vm_config["ram_gb"] * num_workers,
        "dbus_hora": round(dbus_por_hora, 2),
        "costo_vm_hora": round(costo_vm_hora, 2),
        "costo_dbu_hora": round(costo_dbu_hora, 2),
        "costo_total_hora": round(costo_total_hora, 2),
        "costo_mensual": round(costo_mensual, 2),
        "tipo_cluster": tipo_cluster,
        "usa_spot": usar_spot
    }

# 💡 ESTRATEGIAS DE OPTIMIZACIÓN:
# 1. Usar "jobs" clusters en lugar de "all_purpose" cuando sea posible (60% ahorro)
# 2. Habilitar Spot VMs para workloads no críticos (40% ahorro)
# 3. Configurar auto-termination (ej: 30 min de inactividad)
# 4. Usar autoscaling (min/max workers) para ajustar capacidad dinámicamente
# 5. Programar jobs en horarios de baja demanda si es posible

In [ ]:
# Escenario 1: Desarrollo - All-Purpose con autoscaling
dev_config = calcular_costo_cluster(
    vm_type="Standard_DS3_v2",
    num_workers=3,  # Promedio entre 2-4
    horas_mes=160,  # 8 hrs/día * 20 días
    tipo_cluster="all_purpose"
)

print("📊 Escenario 1: Desarrollo (All-Purpose)")
print(f"  VM: {dev_config['vm_type']}")
print(f"  Workers: {dev_config['workers']}")
print(f"  Cores: {dev_config['cores_total']}")
print(f"  RAM: {dev_config['ram_total_gb']} GB")
print(f"  DBUs/hora: {dev_config['dbus_hora']}")
print(f"  💵 Costo/hora: ${dev_config['costo_total_hora']}")
print(f"  💵 Costo/mes: ${dev_config['costo_mensual']}")
print()

In [ ]:
# Escenario 2: Producción - Job Cluster con Spot
prod_config = calcular_costo_cluster(
    vm_type="Standard_DS4_v2",
    num_workers=6,
    horas_mes=180,  # Jobs nocturnos ~6 hrs/día
    tipo_cluster="jobs",
    usar_spot=True
)

print("📊 Escenario 2: Producción (Job + Spot)")
print(f"  VM: {prod_config['vm_type']}")
print(f"  Workers: {prod_config['workers']} (con Spot Instances)")
print(f"  Cores: {prod_config['cores_total']}")
print(f"  RAM: {prod_config['ram_total_gb']} GB")
print(f"  DBUs/hora: {prod_config['dbus_hora']}")
print(f"  💵 Costo/hora: ${prod_config['costo_total_hora']}")
print(f"  💵 Costo/mes: ${prod_config['costo_mensual']}")
print()

# Comparación
ahorro = dev_config['costo_mensual'] - prod_config['costo_mensual']
porcentaje = (ahorro / dev_config['costo_mensual']) * 100

print(f"💡 Ahorro con Job + Spot vs All-Purpose: ${ahorro:.2f}/mes ({porcentaje:.1f}%)")

## Resumen del Laboratorio

### Competencias Adquiridas:

**Gestión de Recursos:**
- Evaluación de autoscaling con cargas de trabajo variables
- Análisis de métricas de performance en Spark UI

**Optimización de Costos:**
- Cálculo de costos de VMs Azure y DBUs de Databricks
- Comparación de escenarios: All-Purpose vs Jobs clusters
- Estrategias de ahorro con Spot instances

**Mejores Prácticas:**
- Dimensionamiento apropiado de clusters según workload
- Trade-offs entre performance y costo
- Configuración de auto-termination y autoscaling

### Próximos Pasos:
- **Lab 03**: Notebooks avanzados con widgets y magic commands
- **Lab 04**: Arquitectura de datos Medallion
- **Lab 05**: Orquestación de workflows multi-task